# Exploración de las fuentes de datos

Este notebook revisa las dos fuentes de ventas **tal como llegan**, antes de cualquier limpieza.
Para cada problema de calidad se indica qué decisión se tomó y en qué parte del pipeline se aplica
(`dags/etl/transformacion.py`).

| Fuente | Archivo en el repo | Origen |
|---|---|---|
| Ventas diarias | `data/raw/ventas_diarias.csv.gz` | Kaggle `carrie1/ecommerce-data`, archivo `data.csv` |
| Histórico | `data/raw/historico_ventas.csv.gz` | Kaggle, archivo `Sales Transaction v.4a.csv` |

In [1]:
import pandas as pd

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 160)

# Se lee todo como texto para ver los datos exactamente como vienen.
diaria = pd.read_csv("../data/raw/ventas_diarias.csv.gz", dtype=str, encoding="latin-1")
historica = pd.read_csv("../data/raw/historico_ventas.csv.gz", dtype=str, encoding="latin-1")

print("Ventas diarias:", diaria.shape)
print("Histórico:     ", historica.shape)

Ventas diarias: (541909, 8)
Histórico:      (536350, 8)


## 1. Estructura de columnas

Las dos fuentes traen la misma información, pero con nombres de columna distintos.

In [2]:
diaria.head(3)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850,United Kingdom


In [3]:
historica.head(3)

,TransactionNo,Date,ProductNo,ProductName,Price,Quantity,CustomerNo,Country
0,581482,12/9/2019,22485,Set Of 2 Wooden Market Crates,21.47,12,17490,United Kingdom
1,581475,12/9/2019,22596,Christmas Star Wish List Chalkboard,10.65,36,13069,United Kingdom
2,581475,12/9/2019,23235,Storage Tin Vintage Leaf,11.53,12,13069,United Kingdom


In [4]:
# Equivalencia de columnas que usa el pipeline (MAPEO_COLUMNAS en dags/etl/config.py).
pd.DataFrame({
    "ventas_diarias": diaria.columns,
    "historico_ventas": historica.columns,
    "esquema_comun": ["factura", "codigo_producto", "descripcion", "cantidad",
                      "fecha", "precio_unitario", "cliente_id", "pais"],
})

,ventas_diarias,historico_ventas,esquema_comun
0,InvoiceNo,TransactionNo,factura
1,StockCode,Date,codigo_producto
2,Description,ProductNo,descripcion
3,Quantity,ProductName,cantidad
4,InvoiceDate,Price,fecha
5,UnitPrice,Quantity,precio_unitario
6,CustomerID,CustomerNo,cliente_id
7,Country,Country,pais


**Diferencias de formato encontradas:**
- La fecha de la fuente diaria trae hora (`12/1/2010 8:26`); la del histórico solo el día (`12/9/2019`).
- Las descripciones de la fuente diaria vienen en mayúsculas; las del histórico en formato título.

**Decisión:** cada fuente se lleva a un esquema común con su propio mapeo de columnas y formato de fecha
(`estandarizar_fuente`).

## 2. Valores nulos

In [5]:
pd.DataFrame({
    "nulos_ventas_diarias": diaria.isna().sum().values,
    "nulos_historico": historica.isna().sum().values,
}, index=["factura", "codigo_producto", "descripcion", "cantidad", "fecha", "precio_unitario", "cliente_id", "pais"])

,nulos_ventas_diarias,nulos_historico
factura,0,0
codigo_producto,0,0
descripcion,1454,0
cantidad,0,0
fecha,0,0
precio_unitario,0,0
cliente_id,135080,55
pais,0,0


In [6]:
sin_cliente = diaria["CustomerID"].isna()
print(f"Líneas sin customer ID en ventas diarias: {sin_cliente.sum():,} ({sin_cliente.mean():.1%})")
print(f"Facturas afectadas: {diaria.loc[sin_cliente, 'InvoiceNo'].nunique():,}")

Líneas sin customer ID en ventas diarias: 135,080 (24.9%)
Facturas afectadas: 3,710


**Decisión (caso ambiguo):** las líneas sin customer ID se **incluyen** con el cliente `DESCONOCIDO`.
Son una cuarta parte de las líneas; excluirlas haría perder ese revenue en todos los reportes.
Además permite comparar su comportamiento con el de los clientes identificados (pregunta 5).

## 3. Tipos de dato y rangos

In [7]:
for nombre, df, col_fecha, formato, col_precio in [
    ("ventas_diarias", diaria, "InvoiceDate", "%m/%d/%Y %H:%M", "UnitPrice"),
    ("historico_ventas", historica, "Date", "%m/%d/%Y", "Price"),
]:
    fechas = pd.to_datetime(df[col_fecha], format=formato)
    print(f"{nombre}: fechas de {fechas.min()} a {fechas.max()}")

pd.DataFrame({
    "cantidad_diaria": pd.to_numeric(diaria["Quantity"]).describe(),
    "precio_diaria": pd.to_numeric(diaria["UnitPrice"]).describe(),
    "cantidad_historico": pd.to_numeric(historica["Quantity"]).describe(),
    "precio_historico": pd.to_numeric(historica["Price"]).describe(),
}).round(2)

ventas_diarias: fechas de 2010-12-01 08:26:00 a 2011-12-09 12:50:00


historico_ventas: fechas de 2018-12-01 00:00:00 a 2019-12-09 00:00:00


,cantidad_diaria,precio_diaria,cantidad_historico,precio_historico
count,541909.00,541909.00,536350.00,536350.00
mean,9.55,4.61,9.92,12.66
std,218.08,96.76,216.66,8.49
min,-80995.00,-11062.06,-80995.00,5.13
25%,1.00,1.25,1.00,10.99
50%,3.00,2.08,3.00,11.94
75%,10.00,4.13,10.00,14.09
max,80995.00,38970.00,80995.00,660.62


Hay cantidades extremas (±80.995) y precios negativos. Se revisan en las secciones siguientes.
Las fechas están en hora local de la tienda (Reino Unido). **Decisión:** se convierten de `Europe/London`
a UTC (`convertir_a_utc`); la zona horaria de origen es la Airflow Variable `zona_horaria_origen`.

## 4. Duplicados exactos

In [8]:
print(f"Duplicados exactos en ventas diarias: {diaria.duplicated().sum():,}")
print(f"Duplicados exactos en histórico:      {historica.duplicated().sum():,}")

Duplicados exactos en ventas diarias: 5,268


Duplicados exactos en histórico:      5,200


**Decisión:** se conserva la primera aparición y las repeticiones van a `log_rechazos`.

## 5. Cantidades negativas: devoluciones y ajustes

In [9]:
cantidad = pd.to_numeric(diaria["Quantity"])
precio = pd.to_numeric(diaria["UnitPrice"])
es_cancelacion = diaria["InvoiceNo"].str.startswith("C")

pd.Series({
    "Líneas con cantidad <= 0": (cantidad <= 0).sum(),
    "  de ellas en facturas que empiezan con C (cancelaciones)": ((cantidad <= 0) & es_cancelacion).sum(),
    "  de ellas sin C (ajustes de inventario)": ((cantidad <= 0) & ~es_cancelacion).sum(),
    "  ajustes con precio 0": ((cantidad <= 0) & ~es_cancelacion & (precio == 0)).sum(),
}, name="ventas_diarias")

Líneas con cantidad <= 0                                     10624
  de ellas en facturas que empiezan con C (cancelaciones)     9288
  de ellas sin C (ajustes de inventario)                      1336
  ajustes con precio 0                                        1336
Name: ventas_diarias, dtype: int64

**Regla de negocio:** toda línea con cantidad menor o igual a cero se separa de las ventas y va a
`fact_devoluciones`, con tipo `CANCELACION` (factura con C) o `AJUSTE` (el resto). Así se puede calcular
el neto por producto y por periodo.

## 6. Precios cero o negativos

In [10]:
venta = cantidad > 0
pd.Series({
    "Ventas (cantidad > 0) con precio <= 0": (venta & (precio <= 0)).sum(),
    "  de ellas sin customer ID": (venta & (precio <= 0) & sin_cliente).sum(),
    "Líneas con precio negativo": (precio < 0).sum(),
}, name="ventas_diarias")

Ventas (cantidad > 0) con precio <= 0    1181
  de ellas sin customer ID               1141
Líneas con precio negativo                  2
Name: ventas_diarias, dtype: int64

In [11]:
# Las líneas con precio negativo son ajustes contables de deuda, no ventas.
diaria[precio < 0]

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
299983,A563186,B,Adjust bad debt,1,8/12/2011 14:51,-11062.06,NaN,United Kingdom
299984,A563187,B,Adjust bad debt,1,8/12/2011 14:52,-11062.06,NaN,United Kingdom


**Regla de negocio:** una venta con precio cero o negativo se rechaza y queda en `log_rechazos` con el motivo.

## 7. Códigos de producto fuera del patrón del catálogo

In [12]:
codigos = diaria["StockCode"].str.upper().str.replace(r"\s+", "", regex=True)
fuera_de_patron = ~codigos.str.match(r"^\d{5}[A-Z]*$")

print(f"Códigos con minúsculas: {diaria['StockCode'].str.contains('[a-z]').sum():,} líneas")
print(f"Líneas con códigos fuera del patrón '5 dígitos + letras opcionales': {fuera_de_patron.sum():,}")
codigos[fuera_de_patron].value_counts().head(15)

Códigos con minúsculas: 1,977 líneas
Líneas con códigos fuera del patrón '5 dígitos + letras opcionales': 2,995


StockCode
POST            1256
DOT              710
M                572
C2               144
D                 77
S                 63
BANKCHARGES       37
AMAZONFEE         34
CRUK              16
DCGSSGIRL         13
DCGSSBOY          11
GIFT_0001_20      10
GIFT_0001_10       9
GIFT_0001_30       8
DCGS0003           5
Name: count, dtype: int64

In [13]:
# Qué representan esos códigos, según su descripción más frecuente.
diaria[fuera_de_patron].groupby(codigos[fuera_de_patron])["Description"].agg(
    lambda s: s.mode().iat[0] if not s.mode().empty else None).head(15)

StockCode
AMAZONFEE                      AMAZON FEE
B                         Adjust bad debt
BANKCHARGES                  Bank Charges
C2                               CARRIAGE
CRUK                      CRUK Commission
D                                Discount
DCGS0003              BOXED GLASS ASHTRAY
DCGS0004       HAYNES CAMPER SHOULDER BAG
DCGS0055                             None
DCGS0057                             None
DCGS0066P                            None
DCGS0067                             ebay
DCGS0068                             ebay
DCGS0069            OOH LA LA DOGS COLLAR
DCGS0070            CAMOUFLAGE DOG COLLAR
Name: Description, dtype: object

Son cargos y movimientos administrativos (envío, comisiones de Amazon, cargos bancarios, ajustes manuales,
donaciones), no productos.

**Decisión:** primero se normaliza el código (mayúsculas, sin espacios), como pide el negocio. Luego,
los códigos que no siguen el patrón `5 dígitos + letras opcionales` se rechazan con su motivo.

## 8. Descripciones inconsistentes para un mismo código

In [14]:
con_descripcion = diaria.dropna(subset=["Description"])
variantes = con_descripcion.groupby(codigos)["Description"].nunique()
variantes_normalizadas = con_descripcion.groupby(codigos)["Description"].agg(
    lambda s: s.str.upper().str.strip().str.replace(r"\s+", " ", regex=True).nunique())

print(f"Códigos con más de una descripción: {(variantes > 1).sum():,}")
print(f"Siguen con más de una después de normalizar mayúsculas y espacios: {(variantes_normalizadas > 1).sum():,}")

Códigos con más de una descripción: 645
Siguen con más de una después de normalizar mayúsculas y espacios: 638


In [15]:
# Ejemplos: además de variaciones de escritura, hay notas internas del almacén
# ("check", "damages", "wet pallet") escritas en el campo de descripción.
ejemplos = con_descripcion[codigos.loc[con_descripcion.index].isin(["85175", "23131", "22900"])]
ejemplos.groupby("StockCode")["Description"].value_counts()

StockCode  Description                     
22900       SET 2 TEA TOWELS I LOVE LONDON     282
           SET 2 TEA TOWELS I LOVE LONDON      282
23131      MISTLETOE HEART WREATH CREAM        114
           MISELTOE HEART WREATH WHITE           4
           ?                                     1
           MISELTOE HEART WREATH CREAM           1
           had been put aside                    1
85175      CACTI T-LIGHT CANDLES               148
           ? sold as sets?                       1
           Amazon sold sets                      1
           check                                 1
           dotcom sold sets                      1
           wrongly sold sets                     1
Name: count, dtype: int64

**Decisión (caso ambiguo):** el nombre canónico de cada código es la descripción normalizada
(mayúsculas, sin espacios sobrantes) **que más se repite**. Las notas del almacén aparecen pocas veces,
así que nunca ganan. En caso de empate se toma la primera en orden alfabético (`nombre_canonico`).
La descripción original se conserva en las tablas de hechos para poder auditar las variantes.

## 9. Distribución por país

In [16]:
diaria["Country"].value_counts().head(10).to_frame("lineas")

,lineas
Country,
United Kingdom,495478
Germany,9495
France,8557
EIRE,8196
Spain,2533
Netherlands,2371
Belgium,2069
Switzerland,2002
Portugal,1519


Las dos fuentes son de una tienda del Reino Unido y los precios están en libras esterlinas (GBP).
**Decisión:** todos los montos se guardan en GBP; no hay conversión de moneda.

## 10. Comparación entre las dos fuentes (duplicados entre fuentes)

El enunciado dice que el histórico contiene años adicionales. Antes de cargarlo se compara contra la
fuente diaria.

In [17]:
facturas_comunes = set(diaria["InvoiceNo"]) & set(historica["TransactionNo"])
print(f"Facturas en ventas diarias: {diaria['InvoiceNo'].nunique():,}")
print(f"Facturas en histórico:      {historica['TransactionNo'].nunique():,}")
print(f"Facturas en ambas:          {len(facturas_comunes):,}")

clave_diaria = set(zip(diaria["InvoiceNo"], codigos))
clave_historica = list(zip(historica["TransactionNo"], historica["ProductNo"].str.upper().str.strip()))
coinciden = sum(clave in clave_diaria for clave in clave_historica)
print(f"Líneas del histórico con la misma factura y producto que ventas diarias: {coinciden:,} de {len(historica):,}")

Facturas en ventas diarias: 25,900
Facturas en histórico:      23,204
Facturas en ambas:          23,190


Líneas del histórico con la misma factura y producto que ventas diarias: 534,539 de 536,350


In [18]:
# Para las facturas comunes: ¿cuántos días hay entre la fecha de una fuente y la de la otra?
fecha_diaria = (diaria.drop_duplicates("InvoiceNo").set_index("InvoiceNo")["InvoiceDate"]
                .pipe(pd.to_datetime, format="%m/%d/%Y %H:%M").dt.normalize())
fecha_historica = (historica.drop_duplicates("TransactionNo").set_index("TransactionNo")["Date"]
                   .pipe(pd.to_datetime, format="%m/%d/%Y"))
comunes = sorted(facturas_comunes)
(fecha_historica[comunes] - fecha_diaria[comunes]).dt.days.value_counts().to_frame("facturas")

,facturas
2922,23190


In [19]:
# Mismo producto y factura, pero ¿mismo precio?
comparacion = diaria.merge(historica, left_on=["InvoiceNo", "StockCode", "Quantity"],
                           right_on=["TransactionNo", "ProductNo", "Quantity"])
comparacion[["InvoiceNo", "StockCode", "Description", "ProductName", "UnitPrice", "Price"]].head()

,InvoiceNo,StockCode,Description,ProductName,UnitPrice,Price
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,Cream Hanging Heart T-Light Holder,2.55,12.86
1,536365,71053,WHITE METAL LANTERN,White Moroccan Metal Lantern,3.39,13.72
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,Cream Cupid Hearts Coat Hanger,2.75,13.07
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,Knitted Union Flag Hot Water Bottle,3.39,13.72
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,Red Woolly Hottie White Heart,3.39,13.72


**Hallazgo:** el histórico **no** son años adicionales. Son las mismas facturas de la fuente diaria
(mismo número, producto, cantidad y cliente) con todas las fechas corridas exactamente 2.922 días (8 años)
y los precios alterados. Cargarlo como ventas nuevas duplicaría el revenue con precios inflados.

**Decisión (caso ambiguo):**
1. Una línea del histórico es un duplicado si tiene **la misma factura y el mismo código de producto** que
   una línea de la fuente diaria. No se usan la fecha ni el precio porque el histórico los trae alterados.
   **Gana la fuente diaria**, que es la operacional.
2. Las pocas líneas del histórico sin pareja tienen fechas de 2019, posteriores a la fuente diaria. Un
   histórico debe ser anterior, así que se rechazan por fecha inconsistente.

Todas quedan en `log_rechazos` con su motivo: ningún registro se pierde sin explicación.

**Otra opción evaluada:** el evaluador ofreció también el archivo `online_retail.csv` del enlace
`thedevastator/online-retail-transaction-dataset`. Se descartó porque es una copia exacta de `data.csv`
(mismas 541.909 filas y mismas fechas, solo con una columna `index` adicional y los IDs como decimal), así que
no permitía demostrar la unificación de columnas y formatos distintos que describe el enunciado.

## 11. Resumen de decisiones

| Problema | Decisión | Dónde se aplica |
|---|---|---|
| Columnas y formatos de fecha distintos | Mapeo por fuente a un esquema común | `estandarizar_fuente` |
| Fechas en hora local | Conversión de Europe/London a UTC | `convertir_a_utc` |
| Códigos con minúsculas o espacios | Mayúsculas y sin espacios | `normalizar_codigo` |
| Duplicados exactos | Rechazo (se conserva el primero) | `transformar`, paso 2 |
| Histórico duplicado de la fuente diaria | Rechazo; gana la fuente diaria | `transformar`, paso 3 |
| Histórico con fechas posteriores | Rechazo por fecha inconsistente | `transformar`, paso 3 |
| Códigos que no son productos | Rechazo | `transformar`, paso 4 |
| Cantidad <= 0 | Separada como devolución o ajuste | `transformar`, paso 5 |
| Venta con precio <= 0 | Rechazo | `transformar`, paso 6 |
| Sin customer ID | Se incluye como `DESCONOCIDO` | `transformar` |
| Varias descripciones por código | Nombre canónico = la más frecuente | `nombre_canonico` |